# Adversarial Search: Playing Connect 4

Student Name: [Add your name]

I have used the following AI tools: [list tools]

I understand that my submission needs to be my own work: [your initials]

## Learning Outcomes

* Implement adversarial search algorithms for strategic game play.
* Analyze and optimize search in complex game spaces.
* Design effective heuristic evaluation functions.
* Compare performance across different agent strategies.
* Evaluate algorithmic trade-offs between decision quality and efficiency.

## Instructions

Total Points: Undergraduates 100, graduate students 110

Complete this notebook and submit it. The notebook needs to be a complete project report with your implementation, documentation including a short discussion of how your implementation works and your design choices, and experimental results (e.g., tables and charts with simulation results) with a short discussion of what they mean. Use the provided notebook cells and insert additional code and markdown cells as needed. Submit the completely rendered notebook as a HTML file. 


## Introduction

You will implement different versions of agents that play Connect 4:

> "Connect 4 is a two-player connection board game, in which the players choose a color and then take turns dropping colored discs into a seven-column, six-row vertically suspended grid. The pieces fall straight down, occupying the lowest available space within the column. The objective of the game is to be the first to form a horizontal, vertical, or diagonal line of four of one's own discs." (see [Connect Four on Wikipedia](https://en.wikipedia.org/wiki/Connect_Four))

Note that [Connect-4 has been solved](https://en.wikipedia.org/wiki/Connect_Four#Mathematical_solution)
in 1988. A connect-4 solver with a discussion of how to solve different parts of the problem can be found here: https://connect4.gamesolver.org/en/

In [1]:
# change directory to Google Drive for Google Colab
try:
    from google.colab import drive
    import os

    drive.mount('/content/drive')
    os.chdir('/content/drive/My Drive/Colab Notebooks/')
    print("Your working directory is now Google Drive: My Drive > Colab Notebooks")
except ImportError:
    pass

## Task 1: Defining the Search Problem [10 point]

**State.** A state is the board (a two-dimensional NumPy integer array) together with the player whose turn it is. Cells are `0` (empty), `1` (player 1), or `-1` (player -1). The initial state is an empty 6-by-7 board with player 1 to move. The board alone does not encode whose turn it is; search code keeps that player separately.

**Actions.** `actions(state)` returns the column indices whose top cells are empty. These are exactly the columns into which another disc can legally fall; terminal states have no actions.

**Transition model.** `result(state, player, action)` copies the board, places `player` in the lowest empty row of `action`, and returns the new board. It never mutates the input. The caller changes the side to move after each transition.

**Terminal test and utility.** `terminal(state)` is true when either player has four consecutive discs horizontally, vertically, or diagonally, or when the board is full. `utility(state, player)` returns `+1` for a win by `player`, `-1` for the opponent's win, and `0` for a draw. It returns `None` for a non-terminal state because no terminal payoff has been determined yet.

The notebook first calls this a ?character array?, but then specifies players as `1` and `-1`, and its starter code creates numeric zeroes. This implementation follows the explicit numeric player encoding and starter code, using an integer NumPy array so the search modules can compare player values directly.

In [2]:
from connect4_core import actions, empty_board, result, terminal, utility

How big is the state space? Give an estimate and explain it.

For the standard board there are 42 cells, so ignoring gravity and turn order gives the loose bound `3^42 ? 1.09 ? 10^20` assignments. A more informative upper bound counts only stacked columns (each column has height 0 through 6) and positions whose counts are consistent with player 1 moving first. Let `a_k` be the coefficient of `x^k` in `(1 + x + x^2 + ... + x^6)^7`. Then the number of such layouts is at most

`sum(k=0..42) a_k * C(k, ceil(k/2)) ? 7.07 ? 10^13`.

This is still an upper bound: it includes boards that cannot occur because the game would already have ended on an earlier move. The exact reachable state count is smaller. A state also includes the side to move, which is determined by the number of discs for non-terminal states.

How big is the game tree that minimax search will go through? Give an estimate and explain it.

A game has at most 42 moves and at most 7 choices at any move. Thus a deliberately loose upper bound for a full-width minimax tree is

`1 + 7 + 7^2 + ... + 7^42 = (7^43 - 1) / 6 ? 3.64 ? 10^35` nodes.

This bound acts as if all seven columns remain available at every depth and ignores wins, so the real tree is smaller. It is nevertheless enormous. Unlike the state space, a game tree counts paths: the same board may be reached by playing independent columns in a different order, so it can appear in multiple tree nodes. Terminal detection and alpha-beta pruning reduce the work, but exhaustive minimax on the standard board is not practical.

## Task 2: Game Environment and Random Agent [25 point]

Use a numpy character array as the board.

In [3]:
import numpy as np
from connect4_core import empty_board

print(empty_board())

[[0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0]]


The standard board is $6 \times 7$ but you can use smaller boards to test your code. Instead of colors (red and yellow), I use 1 and -1 to represent the players. Make sure that your agent functions all have the from: `agent_type(board, player = 1)`, where board is the current board position (in the format above) and player is the player whose next move it is and who the agent should play (as 1 and -1).

In [4]:
# Visualization code by Randolph Rankin

import matplotlib.pyplot as plt

def visualize(board):
    plt.axes()
    rectangle=plt.Rectangle((-0.5,len(board)*-1+0.5),len(board[0]),len(board),fc='blue')
    circles=[]
    for i,row in enumerate(board):
        for j,val in enumerate(row):
            color='white' if val==0 else 'red' if val==1 else 'yellow'
            circles.append(plt.Circle((j,i*-1),0.4,fc=color))

    plt.gca().add_patch(rectangle)
    for circle in circles:
        plt.gca().add_patch(circle)

    plt.axis('scaled')
    plt.show()
    
board = [[0, 0, 0, 0, 0, 0, 0],
         [0, 0, 0, 0, 0, 0, 0],
         [0, 0, 0, 0, 0, 0, 0],
         [0, 0, 0, 1, 0, 0, 0],
         [0, 0, 0, 1, 0, 0, 0],
         [0,-1,-1, 1,-1, 0, 0]]
visualize(board)

Implement helper functions for:

* A check for available actions in each state `actions(state)`.
* The transition model `result(state, player, action)`.
* Check for terminal states `terminal(state)`.
* The utility function `utility(state, player)`.

The player argument is used so your agent can play red or yellow. Make sure that all these functions work with boards of different sizes (number of columns and rows). You can follow the [tic-tac-toe example from class](tictactoe_definitions.ipynb).

**Input and error behavior:** boards must be non-empty, two-dimensional NumPy integer arrays containing only `-1`, `0`, and `1`; `player` must be `1` or `-1`; and `action` must be an in-range integer column. Invalid boards/players/actions raise `TypeError` or `ValueError`. A move on a full column or terminal board raises `ValueError`. The input board is not changed. `actions` returns an empty list in terminal states; `utility` returns `None` while play is ongoing and raises `ValueError` if an invalid board shows winning lines for both players.

Keep `connect4_core.py` in the same directory as this notebook so the imports work in Jupyter.

In [5]:
from connect4_core import actions, result, terminal, utility

Implement an agent that plays randomly. Make sure the agent function receives as the percept the board and returns a valid action. Use an agent function definition with the following signature (arguments):

`def random_player(board, player = 1): ...`

The argument `player` is used for agents that do not store what color they are playing. The value passed on by the environment should be 1 ot -1 for player red and yellow, respectively.  See [Experiments section for tic-tac-toe](tictactoe_and_or_tree_search.ipynb#Experiments) for an example.

In [6]:
from connect4_core import random_player

Let two random agents play 1,000 games. The experiment below alternates which player moves first (500 games each) to balance the first-move position. It reports wins by board player (`1` or `-1`), not by the function name, because both agents use the same policy. Each game is checked to have reached a terminal state.

In [7]:
from collections import Counter

number_of_games = 1000
np.random.seed(20261108)
results = Counter({1: 0, -1: 0, 0: 0})
first_moves = Counter({1: 0, -1: 0})
first_player_wins = 0

for game_index in range(number_of_games):
    board = empty_board()
    first_player = 1 if game_index % 2 == 0 else -1
    first_moves[first_player] += 1
    player = first_player

    while not terminal(board):
        action = random_player(board, player)
        board = result(board, player, action)
        if terminal(board):
            break
        player = -player

    if not terminal(board):
        raise RuntimeError(f"game {game_index} ended without a terminal state")
    game_result = utility(board, 1)
    results[game_result] += 1
    if game_result == first_player:
        first_player_wins += 1

assert sum(results.values()) == number_of_games
assert first_moves[1] == first_moves[-1] == number_of_games // 2

print(f"Games: {number_of_games}")
for outcome, label in ((1, "Player 1 wins"), (-1, "Player -1 wins"), (0, "Draws")):
    print(f"{label}: {results[outcome]} ({results[outcome] / number_of_games:.1%})")
print(f"First player: {first_player_wins} ({first_player_wins / number_of_games:.1%})")
print(f"Starts: player 1 = {first_moves[1]}, player -1 = {first_moves[-1]}")

Games: 1000
Player 1 wins: 510 (51.0%)
Player -1 wins: 487 (48.7%)
Draws: 3 (0.3%)
First player: 546 (54.6%)
Starts: player 1 = 500, player -1 = 500


### Random vs. Random results

Using seed `20261108`, the 1,000-game run produced:

| Outcome | Games | Rate |
|---|---:|---:|
| Player 1 wins | 510 | 51.0% |
| Player -1 wins | 487 | 48.7% |
| Draws | 3 | 0.3% |

Each side started 500 times. The first player won 546 games (54.6%), which is consistent with a modest first-move advantage in this run. Both agents use the same random policy, so the difference is about move order rather than agent strength; one seeded run is evidence, not proof of a statistically significant advantage. The totals sum to exactly 1,000, and the simulation checks every final board is terminal.

## Task 3: Minimax Search with Alpha-Beta Pruning

### Implement the Search [10 points] 

Implement minimax search starting from a given board for specifying the player.

__Important Notes:__ 
* You can reuse code from the [tic-tac-toe examples](.).
* Make sure that all your agent functions have a signature consistent with the random agent above and that it [uses a class to store state information.](store_agent_state_information.ipynb)
This is essential to be able play against agents from other students later.
* The game tree for a $6 \times 7$ board is huge and optimal algorithms need to visit each or a large percentage of all nodes in the tree. You can experiment with smaller boards like a $4 \times 4$ board first.

**Cách cài.** Ý tưởng lấy từ `tictactoe_alpha_beta_tree_search.ipynb`; chỉ đổi: người chơi là `1` / `-1` (đối thủ của `player` là `-player`).

* **Minimax:** thử từng cột còn trống, giả sử đối thủ luôn đáp trả tốt nhất *cho đối thủ*, rồi chọn cột cho mình điểm cao nhất (`+1` thắng, `0` hòa, `-1` thua, nhìn từ `player`). Mỗi thế cờ phải xem qua là một **nút**.
* **Alpha-Beta:** Minimax có thêm "điểm tốt nhất mình đã chắc chắn có"; thấy nhánh nào mà đối thủ làm mình tệ hơn mức đó thì bỏ cả nhánh (**cắt tỉa**). Kết quả không đổi, số nút ít hơn.
* **Move ordering:** thứ tự thử các cột; xếp nước tốt lên trước thì cắt tỉa được nhiều hơn (xem mục Move ordering).
* Giá trị không giảm theo độ sâu, nên thắng sớm hay muộn đều là `+1` (máy giữ cách thắng đầu tiên tìm ra).
* Hàm tìm kiếm nhận `node_limit` (vượt thì dừng và báo "không xong trong giới hạn") và `verbose`; trả về `{"move", "value", "nodes"}`, mỗi lần tìm có bộ đếm nút riêng.
* Tác tử là class `AlphaBetaAgent` với `act(board, player = 1)`, lưu `known_moves`, `move_times`, `nodes` (theo `store_agent_state_information.ipynb`); gặp lại thế cờ đã tính thì trả luôn nước đã nhớ (đúng, vì nước ở gốc chỉ phụ thuộc vào thế cờ).
* `actions, result, terminal, utility, random_player` lấy từ `connect4_core.py` của Tú.

**Nhận xét.** `Search.visit` gộp hai bước max và min của notebook Tic-Tac-Toe thành một: đếm nút, trả về `utility` nếu ván đã kết thúc, rồi thử từng cột theo `move_order`; nút của `player` cập nhật `alpha` và cắt khi `v >= beta`, nút của đối thủ làm ngược lại. Minimax là `Search` với `prune = False`. Cài đặt được kiểm bằng 11 test trong `test_connect4_minimax.py` và bằng các bảng bên dưới.

In [8]:
import inspect
import platform
import sys
import time

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import connect4_minimax
from connect4_core import actions, empty_board, random_player, result, terminal, utility
from connect4_minimax import (AlphaBetaAgent, SearchLimitReached, alpha_beta_search, board_from_moves,
                              center_first, minimax_search, no_ordering, play, timed_search, win_block_center)

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 20)


# Máy chạy notebook này (thời gian bên dưới đo trên máy này, máy khác sẽ khác)
print("Python  :", sys.version.split()[0])
print("NumPy   :", np.__version__)
print("Hệ điều hành:", platform.platform())
print("CPU     :", platform.processor() or platform.machine())

# Thứ tự nước đi ít nút nhất (win_block_center), chọn sau khi xem bảng ở mục "Move ordering" bên dưới.
# Khai báo ở đây để các mục phía trước dùng được. Lưu ý: "ít nút nhất" chưa chắc là "nhanh nhất", xem phần nhận xét.
BEST_ORDER = win_block_center


# Xem lại mã nguồn đã cài trong connect4_minimax.py (chỉ in ra để đọc, không chạy lại thuật toán)
for f in (alpha_beta_search, connect4_minimax.Search, AlphaBetaAgent):
    print(inspect.getsource(f))

Python  : 3.11.18
NumPy   : 2.4.6
Hệ điều hành: Linux-7.0.0-1019-azure-x86_64-with-glibc2.43
CPU     : x86_64
def alpha_beta_search(board, player=1, move_order=no_ordering, node_limit=None, verbose=0):
    """minimax search with alpha-beta pruning. move_order decides in which order the columns are tried.

    Returns {"move", "value", "nodes"}.
    """
    return Search(player, move_order, node_limit, prune=True, verbose=verbose).run(board)

class Search:
    """One search run for ``player``: counts the nodes, enforces the node limit and prints the trace.

    All values are from ``player``'s view (+1 win, 0 draw, -1 loss). With ``prune=False`` it
    is plain minimax; with ``prune=True`` it is minimax with alpha-beta pruning.
    verbose: 0 ... silent, 1 ... print the number of nodes, 2 ... trace each node.
    """

    def __init__(self, player, move_order=no_ordering, node_limit=None, prune=True, verbose=0):
        self.player = player
        self.move_order = move_order
        s

Experiment with some manually created boards (at least 5) to check if the agent spots winning opportunities.

Mình tạo **6 bàn** bằng `board_from_moves`: thắng ngay (hàng ngang, hàng dọc, đường chéo), phải chặn đối thủ, thắng sau 2 lượt (tạo hai đường dọa cùng lúc) và một bàn chuẩn 6×7 gần cuối ván. "Nước mong đợi" được **tính bằng code**, không ghi tay. Cột "Đúng?" là `True` nếu nước chọn nằm trong nhóm mong đợi, hoặc giá trị là `+1` và nước đó vẫn giữ được thắng.

**Nhận xét.**

* Cả 18 dòng (6 bàn × 3 cách chạy) đều "Đúng?" = `True`: máy nhận ra cả cơ hội thắng lẫn nước phải chặn. Bàn 1, 2, 3 và 6 chọn cột thắng ngay; bàn 4 chọn cột 3 để chặn (giá trị 0: hòa nếu hai bên chơi đúng); bàn 5 chọn cột 1, tạo hai đường dọa thắng cùng lúc (giá trị `+1`).
* Minimax và Alpha-Beta luôn cùng giá trị (có `assert`), nhưng Alpha-Beta ít nút hơn nhiều: bàn 4 từ 65,332 xuống 4,733; bàn 2 từ 18,099 xuống 162; bàn 1 từ 3,611 xuống 812.
* `win_block_center` giảm số nút thêm ở 5 trên 6 bàn (bàn 4: 4,733 → 1,278; bàn 1: 812 → 180; bàn 6: 38 → 12), trừ bàn 2 (162 → 167). Nhưng **ở cả 6 bàn nó đều chậm hơn** (bàn 4: 127 ms so với 39 ms; bàn 2: 16 ms so với 1.4 ms), vì mỗi nút phải thử trước từng nước để biết thắng hay chặn.
* Vì giá trị không giảm theo độ sâu, thứ tự mặc định có thể giữ một cách thắng chậm hơn nếu gặp nó trước (giống ghi chú ở notebook Tic-Tac-Toe); `win_block_center` xét nước thắng ngay trước nên tránh được.
* Minimax chạy xong ở cả 6 bàn (dưới 300,000 nút), kể cả bàn 6×7 vì chỉ còn 8 ô trống.

In [9]:
# Helper: bàn cờ ngẫu nhiên (hai bên đi ngẫu nhiên) cho tới khi còn đúng `empty_target` ô trống.
# Nếu ván kết thúc sớm hoặc thế cờ không hợp, tăng seed và thử lại. Trả về cả danh sách nước đi để chạy lại được.
def immediate_wins(board, player):
    """các cột mà player đi vào là thắng ngay."""
    return [a for a in actions(board) if utility(result(board, player, a), player) == 1]


def random_position(shape, empty_target, seed, quiet = False):
    """quiet = True: chỉ nhận thế cờ mà cả hai bên đều chưa có nước thắng ngay."""
    while True:
        np.random.seed(seed)
        board, player, moves = empty_board(shape), 1, []
        while int(np.sum(board == 0)) > empty_target and not terminal(board):
            a = random_player(board, player)
            board, player, moves = result(board, player, a), -player, moves + [a]
        ok = not terminal(board) and int(np.sum(board == 0)) == empty_target
        if ok and quiet:
            ok = not immediate_wins(board, player) and not immediate_wins(board, -player)
        if ok:
            return board, player, moves, seed
        seed += 1


def double_threat_after(board, player, move):
    """sau khi player đi `move`: player có ít nhất 2 cột thắng ngay và đối thủ chưa có cột nào."""
    after = result(board, player, move)
    return (not terminal(after)) and len(immediate_wins(after, player)) >= 2 and not immediate_wins(after, -player)


def expected_moves(board, player):
    """nước mong đợi, TÍNH BẰNG CODE: thắng ngay > chặn đối thủ > tạo hai đường dọa thắng."""
    wins = immediate_wins(board, player)
    if wins: return wins
    blocks = immediate_wins(board, -player)
    if blocks: return blocks
    return [a for a in actions(board) if double_threat_after(board, player, a)]


def keeps_the_win(board, player, move):
    """sau `move`, đối thủ có thua chắc không? (tức là thắng vẫn còn trong tay)"""
    after = result(board, player, move)
    if terminal(after): return utility(after, player) == 1
    return alpha_beta_search(after, -player, move_order = win_block_center)["value"] == -1


# Bàn 6: bàn chuẩn 6x7 gần cuối ván, còn 8 ô trống (hai bên đi ngẫu nhiên, tăng seed từ 20261109 tới khi có nước thắng ngay)
seed6 = 20261109
while True:
    board6, player6, moves6, seed6 = random_position((6, 7), 8, seed6)
    if immediate_wins(board6, player6): break
    seed6 += 1
print("Bàn 6: seed =", seed6, ", các nước đi =", moves6)


# Sáu bàn tự tạo bằng board_from_moves (đi đúng luật từng nước nên bàn luôn hợp lệ).
board_specs = [
    # (tên, kích thước, danh sách nước đi, người cần đi)
    ("1. Thắng ngay (hàng ngang)",       (4, 4), [0, 0, 1, 0, 2, 0], 1),
    ("2. Thắng ngay (hàng dọc)",         (4, 4), [0, 1, 0, 1, 0, 2], 1),
    ("3. Thắng ngay (đường chéo)",       (4, 4), [1, 3, 0, 2, 1, 2, 3, 3, 2, 1], 1),
    ("4. Phải chặn đối thủ",             (4, 4), [0, 0, 1, 0, 2], -1),
    ("5. Thắng sau 2 lượt (2 đường dọa)", (4, 5), [1, 0, 1, 4, 2, 3, 4, 2, 2, 0, 4, 3], 1),
    ("6. Bàn chuẩn 6x7 cuối ván",        (6, 7), moves6, player6),
]

boards = {}
for name, shape, moves, who in board_specs:
    board, player = board_from_moves(shape, moves)
    assert not terminal(board), f"{name}: bàn đã kết thúc"
    assert player == who, f"{name}: không đúng lượt người cần đi"
    boards[name] = (board, player)
    print(name, "| người đi:", player, "| ô trống:", int(np.sum(board == 0)), "| nước mong đợi:", expected_moves(board, player))
    visualize(board)


# Chạy Minimax (giới hạn 300000 nút), Alpha-Beta thứ tự mặc định và Alpha-Beta với win_block_center.
rows = []
for name, (board, player) in boards.items():
    expected = expected_moves(board, player)
    values = {}
    for algo, search, kwargs in [("Minimax", minimax_search, {}),
                                 ("Alpha-Beta (mặc định)", alpha_beta_search, {"move_order": no_ordering}),
                                 ("Alpha-Beta (win_block_center)", alpha_beta_search, {"move_order": win_block_center})]:
        r = timed_search(search, board, player, node_limit = 300000, **kwargs)
        if r["finished"]:
            correct = r["move"] in expected or (r["value"] == 1 and keeps_the_win(board, player, r["move"]))
            values[algo] = r["value"]
        else:
            correct = "—"
        rows.append({"Bàn": name, "Kích thước": f"{board.shape[0]}x{board.shape[1]}", "Ô trống": int(np.sum(board == 0)),
                     "Nước mong đợi": expected, "Thuật toán": algo,
                     "Nước chọn": r["move"] if r["finished"] else "chưa xong",
                     "Giá trị": r["value"] if r["finished"] else "—",
                     "Số nút": r["nodes"] if r["finished"] else f">{r['nodes']}",
                     "Thời gian (ms)": round(r["time_ms"], 1), "Đúng?": correct})
    assert len(set(values.values())) <= 1, f"{name}: các thuật toán cho giá trị khác nhau: {values}"

table_boards = pd.DataFrame(rows)
display(table_boards)
assert all(c in (True, "—") for c in table_boards["Đúng?"]), "có nước đi sai!"
print("Mọi nước đi đều đúng; Minimax và Alpha-Beta luôn cùng giá trị trên các bàn mà Minimax chạy xong.")

Bàn 6: seed = 20261118 , các nước đi = [0, 1, 5, 5, 4, 2, 1, 6, 4, 1, 4, 0, 0, 0, 3, 3, 2, 1, 6, 4, 5, 5, 6, 4, 5, 1, 5, 6, 1, 6, 4, 2, 6, 2]
1. Thắng ngay (hàng ngang) | người đi: 1 | ô trống: 10 | nước mong đợi: [3]
2. Thắng ngay (hàng dọc) | người đi: 1 | ô trống: 10 | nước mong đợi: [0]
3. Thắng ngay (đường chéo) | người đi: 1 | ô trống: 6 | nước mong đợi: [3]


4. Phải chặn đối thủ | người đi: -1 | ô trống: 11 | nước mong đợi: [3]
5. Thắng sau 2 lượt (2 đường dọa) | người đi: 1 | ô trống: 8 | nước mong đợi: [1]
6. Bàn chuẩn 6x7 cuối ván | người đi: 1 | ô trống: 8 | nước mong đợi: [3]


,Bàn,Kích thước,Ô trống,Nước mong đợi,Thuật toán,Nước chọn,Giá trị,Số nút,Thời gian (ms),Đúng?
0,1. Thắng ngay (hàng ngang),4x4,10,[3],Minimax,3,1,3611,29.8,True
1,1. Thắng ngay (hàng ngang),4x4,10,[3],Alpha-Beta (mặc định),3,1,812,6.8,True
2,1. Thắng ngay (hàng ngang),4x4,10,[3],Alpha-Beta (win_block_center),3,1,180,22.0,True
3,2. Thắng ngay (hàng dọc),4x4,10,[0],Minimax,0,1,18099,143.2,True
4,2. Thắng ngay (hàng dọc),4x4,10,[0],Alpha-Beta (mặc định),0,1,162,1.4,True
5,2. Thắng ngay (hàng dọc),4x4,10,[0],Alpha-Beta (win_block_center),0,1,167,20.1,True
6,3. Thắng ngay (đường chéo),4x4,6,[3],Minimax,3,1,200,1.6,True
7,3. Thắng ngay (đường chéo),4x4,6,[3],Alpha-Beta (mặc định),3,1,154,1.2,True
8,3. Thắng ngay (đường chéo),4x4,6,[3],Alpha-Beta (win_block_center),3,1,34,3.5,True
9,4. Phải chặn đối thủ,4x4,11,[3],Minimax,3,0,65332,526.8,True


Mọi nước đi đều đúng; Minimax và Alpha-Beta luôn cùng giá trị trên các bàn mà Minimax chạy xong.


How long does it take to make a move? Start with a smaller board with 4 columns and make the board larger by adding columns/rows. Explain why using this algorithm on a standard $6 \times 7$ board is not feasible.

**Cách đo.** Hai loạt, dùng Alpha-Beta với `no_ordering` (rẻ nhất mỗi nút trên bàn trống và bàn cuối ván, xem mục Move ordering). Mỗi lần tìm giới hạn 500,000 nút; vượt thì ghi "không xong" và bỏ qua các bàn lớn hơn cùng hướng.

* **Loạt 1: bàn trống, tăng dần kích thước:** 4×4, rồi thêm cột (4×5, 4×6, 4×7) hoặc hàng (5×4, 6×4), 5×5 và 6×7.
* **Loạt 2: bàn 6×7, số ô trống tăng dần** (8, 10, ..., 20): hai bên đi ngẫu nhiên có seed, chỉ nhận thế cờ "yên tĩnh" (chưa bên nào có nước thắng ngay, vì khi đó máy tìm rất nhanh và số nút không phản ánh độ khó).

**Nhận xét.**

* **Loạt 1:** 4×4 là kích thước duy nhất xong: Alpha-Beta xét 62,889 nút (khoảng 0.5 giây), còn Minimax thuần chưa xong sau 500,000 nút (khoảng 4 giây). Chỉ thêm 1 cột (4×5) hoặc 1 hàng (5×4) là Alpha-Beta cũng vượt 500,000 nút, tức nhiều hơn ít nhất 8 lần chỉ vì thêm 4 ô; các bàn lớn hơn được "bỏ qua".
* **Loạt 2:** số nút tăng nhanh theo số ô trống: 23 (8 ô), 193, 1,612, 2,206, 1,557, 19,006 (18 ô, khoảng 0.2 giây), rồi **vượt 500,000 nút ở 20 ô trống** (khoảng 4 giây). Đường không đều vì mỗi thế cờ ngẫu nhiên khác nhau, nhưng xu hướng tăng rất nhanh.
* **Ước lượng cho bàn 6×7 trống (42 ô)**, với tốc độ máy này khoảng 125,000 nút/giây (số nút là cố định, thời gian tùy máy). Cách 1, kẻ đường thẳng qua các điểm đã xong: khoảng 6.6 × 10^8 nút, tức khoảng 1.5 giờ, nhưng chắc chắn **quá thấp** vì không thấy bước nhảy lớn ở 20 ô trống. Cách 2, dùng chính bước nhảy 18 → 20 ô (tăng ít nhất 26 lần cho mỗi 2 ô): ít nhất khoảng 2 × 10^21 nút, tức khoảng 5 × 10^8 năm. Cận trên 3.64 × 10^35 nút (Task 1) cỡ 10^23 năm, cố ý rất rộng.
* **Kết luận:** dù lấy con số nhỏ nhất đáng tin (Cách 2), tìm kiếm đầy đủ trên bàn 6×7 trống cũng cần cỡ hàng trăm triệu năm, nên chỉ chạy được trên bàn nhỏ hoặc khi ván gần hết; bàn chuẩn phải cắt độ sâu và dùng hàm đánh giá (Task 4).

In [10]:
# Loạt 1: bàn TRỐNG, tăng dần kích thước (đây là trường hợp nặng nhất: nước đi đầu tiên).
# Thứ tự nước đi: SIZE_ORDER (xem lý do ở phần nhận xét). Giới hạn 500000 nút cho mỗi lần tìm.
SIZE_LIMIT = 500000
SIZE_ORDER = no_ordering
size_rows = []
size_done = {}      # (hàng, cột) -> Alpha-Beta đã xong chưa
mm_done = {}        # (hàng, cột) -> Minimax đã xong chưa


def measure_size(shape, smaller):
    """chạy Minimax và Alpha-Beta trên bàn trống. Bỏ qua nếu một kích thước nhỏ hơn (cùng hướng) đã vượt giới hạn."""
    board = empty_board(shape)
    for algo, search, kwargs, done in [("Minimax", minimax_search, {}, mm_done),
                                       ("Alpha-Beta", alpha_beta_search, {"move_order": SIZE_ORDER}, size_done)]:
        if any(done.get(s) is False for s in smaller):
            done[shape] = False
            size_rows.append({"Kích thước": f"{shape[0]}x{shape[1]}", "Ô trống": shape[0] * shape[1], "Thuật toán": algo,
                              "Xong?": "bỏ qua (chắc chắn còn lâu hơn)", "Số nút": "—", "Thời gian (s)": "—"})
            continue
        r = timed_search(search, board, 1, node_limit = SIZE_LIMIT, **kwargs)
        done[shape] = r["finished"]
        size_rows.append({"Kích thước": f"{shape[0]}x{shape[1]}", "Ô trống": shape[0] * shape[1], "Thuật toán": algo,
                          "Xong?": "xong" if r["finished"] else "không xong trong giới hạn",
                          "Số nút": r["nodes"] if r["finished"] else f">{SIZE_LIMIT}",
                          "Thời gian (s)": round(r["time_ms"] / 1000, 1)})


# thêm cột (giữ 4 hàng): 4x4 -> 4x5 -> 4x6 -> 4x7
measure_size((4, 4), [])
measure_size((4, 5), [(4, 4)])
measure_size((4, 6), [(4, 5)])
measure_size((4, 7), [(4, 6)])
display(pd.DataFrame(size_rows))


# thêm hàng (giữ 4 cột): 4x4 -> 5x4 -> 6x4, rồi 5x5 và 6x7 (chỉ chạy khi các bàn nhỏ hơn đã xong)
measure_size((5, 4), [(4, 4)])
measure_size((6, 4), [(5, 4)])
measure_size((5, 5), [(4, 5), (5, 4)])
measure_size((6, 7), [(5, 5), (4, 7), (6, 4)])
table_sizes = pd.DataFrame(size_rows).drop_duplicates(subset = ["Kích thước", "Thuật toán"], keep = "first")
display(table_sizes)


# Loạt 2: bàn chuẩn 6x7, số ô trống tăng dần. Hai bên đi ngẫu nhiên tới khi còn E ô trống;
# chỉ nhận thế cờ "yên tĩnh" (chưa bên nào có nước thắng ngay), nếu không thì đổi seed. Ghi lại seed đã dùng.
POSITION_LIMIT = 500000
POSITION_ORDER = no_ordering
position_rows = []
positions = {}
for E in [8, 10, 12, 14, 16, 18, 20]:
    board, player, moves, seed = random_position((6, 7), E, 20261109, quiet = True)
    positions[E] = (board, player, moves, seed)
    r = timed_search(alpha_beta_search, board, player, move_order = POSITION_ORDER, node_limit = POSITION_LIMIT)
    position_rows.append({"Ô trống": E, "Seed đã dùng": seed, "Số nút": r["nodes"], "Thời gian (s)": round(r["time_ms"] / 1000, 2),
                          "Xong?": "xong" if r["finished"] else "không xong trong giới hạn"})
    if not r["finished"]:
        break      # các thế cờ nhiều ô trống hơn chắc chắn còn lâu hơn

table_positions = pd.DataFrame(position_rows)
display(table_positions)

done = table_positions[table_positions["Xong?"] == "xong"]
fig, ax = plt.subplots(figsize = (6, 3.8))
ax.plot(done["Ô trống"], done["Số nút"], "o-", color = "tab:blue", label = "đã xong")
not_done = table_positions[table_positions["Xong?"] != "xong"]
ax.plot(not_done["Ô trống"], not_done["Số nút"], "o", mfc = "white", color = "tab:red", label = "vượt giới hạn")
ax.set_yscale("log")
ax.set_xlabel("Số ô trống trên bàn 6x7")
ax.set_ylabel("Số nút đã xét (thang log)")
ax.set_title("Số nút Alpha-Beta tăng theo số ô trống")
ax.grid(True, alpha = 0.3)
ax.legend()
plt.show()


# Ước lượng THÔ cho bàn chuẩn 6x7 (42 ô trống) dựa trên các số đo ở trên.
all_runs = pd.concat([
    pd.DataFrame(size_rows).query("`Xong?` == 'xong'")[["Số nút", "Thời gian (s)"]],
    done[["Số nút", "Thời gian (s)"]]]).astype(float)
speed = all_runs["Số nút"].sum() / all_runs["Thời gian (s)"].sum()          # nút mỗi giây
year = 3600 * 24 * 365
print(f"Tốc độ trung bình: khoảng {speed:,.0f} nút/giây\n")

# Cách 1: đường thẳng khớp trên thang log qua các thế cờ đã xong (bỏ thế cờ quá nhỏ vì nhiễu)
d = done[done["Số nút"] > 100]
fit = np.polyfit(d["Ô trống"], np.log10(d["Số nút"]), 1)
est_fit = 10 ** (fit[0] * 42 + fit[1])
print(f"Cách 1 (đường thẳng qua {len(d)} điểm đã xong): mỗi khi thêm 2 ô trống số nút tăng gấp khoảng {10 ** (2 * fit[0]):.1f} lần;")
print(f"   với 42 ô trống: khoảng {est_fit:.1e} nút, tức khoảng {est_fit / speed / year:.1e} năm.")

# Cách 2: dùng bước nhảy cuối cùng quan sát được (từ thế cờ lớn nhất đã xong tới thế cờ đầu tiên vượt giới hạn)
last_ok, first_fail = done.iloc[-1], not_done.iloc[0]
per_two = first_fail["Số nút"] / last_ok["Số nút"]          # chỉ là cận dưới, vì lần này tìm chưa xong
steps = (42 - first_fail["Ô trống"]) / 2
est_jump = first_fail["Số nút"] * per_two ** steps
print(f"Cách 2 (bước nhảy {int(last_ok['Ô trống'])} -> {int(first_fail['Ô trống'])} ô trống, tăng ít nhất {per_two:.0f} lần cho mỗi 2 ô trống):")
print(f"   với 42 ô trống: ít nhất khoảng {est_jump:.1e} nút, tức khoảng {est_jump / speed / year:.1e} năm.")

bound = 3.64e35
print(f"\nCận trên của cây trò chơi (Task 1): {bound:.2e} nút, tức khoảng {bound / speed / year:.1e} năm.")

,Kích thước,Ô trống,Thuật toán,Xong?,Số nút,Thời gian (s)
0,4x4,16,Minimax,không xong trong giới hạn,>500000,3.9
1,4x4,16,Alpha-Beta,xong,62889,0.5
2,4x5,20,Minimax,bỏ qua (chắc chắn còn lâu hơn),—,—
3,4x5,20,Alpha-Beta,không xong trong giới hạn,>500000,4.1
4,4x6,24,Minimax,bỏ qua (chắc chắn còn lâu hơn),—,—
5,4x6,24,Alpha-Beta,bỏ qua (chắc chắn còn lâu hơn),—,—
6,4x7,28,Minimax,bỏ qua (chắc chắn còn lâu hơn),—,—
7,4x7,28,Alpha-Beta,bỏ qua (chắc chắn còn lâu hơn),—,—


,Kích thước,Ô trống,Thuật toán,Xong?,Số nút,Thời gian (s)
0,4x4,16,Minimax,không xong trong giới hạn,>500000,3.9
1,4x4,16,Alpha-Beta,xong,62889,0.5
2,4x5,20,Minimax,bỏ qua (chắc chắn còn lâu hơn),—,—
3,4x5,20,Alpha-Beta,không xong trong giới hạn,>500000,4.1
4,4x6,24,Minimax,bỏ qua (chắc chắn còn lâu hơn),—,—
5,4x6,24,Alpha-Beta,bỏ qua (chắc chắn còn lâu hơn),—,—
6,4x7,28,Minimax,bỏ qua (chắc chắn còn lâu hơn),—,—
7,4x7,28,Alpha-Beta,bỏ qua (chắc chắn còn lâu hơn),—,—
8,5x4,20,Minimax,bỏ qua (chắc chắn còn lâu hơn),—,—
9,5x4,20,Alpha-Beta,không xong trong giới hạn,>500000,4.1


,Ô trống,Seed đã dùng,Số nút,Thời gian (s),Xong?
0,8,20261330,23,0.00,xong
1,10,20261272,193,0.00,xong
2,12,20261272,1612,0.01,xong
3,14,20261121,2206,0.02,xong
4,16,20261121,1557,0.01,xong
5,18,20261121,19006,0.15,xong
6,20,20261118,500000,4.08,không xong trong giới hạn


Tốc độ trung bình: khoảng 126,791 nút/giây

Cách 1 (đường thẳng qua 5 điểm đã xong): mỗi khi thêm 2 ô trống số nút tăng gấp khoảng 2.5 lần;
   với 42 ô trống: khoảng 6.6e+08 nút, tức khoảng 1.7e-04 năm.
Cách 2 (bước nhảy 18 -> 20 ô trống, tăng ít nhất 26 lần cho mỗi 2 ô trống):
   với 42 ô trống: ít nhất khoảng 2.1e+21 nút, tức khoảng 5.2e+08 năm.

Cận trên của cây trò chơi (Task 1): 3.64e+35 nút, tức khoảng 9.1e+22 năm.


### Move ordering [5 points]

Starting the search with better moves will increase the efficiency of alpha-beta pruning. Describe and implement a simple move ordering strategy. Make a table that shows how the ordering strategies influence the time it takes to make a move?

Mình thử ba cách xếp thứ tự các cột (trong `connect4_minimax.py`, truyền qua tham số `move_order`):

* **`no_ordering`:** từ trái sang phải, đúng thứ tự của `actions()`.
* **`center_first`:** cột gần giữa trước (cột giữa nằm trong nhiều đường bốn quân nhất); bằng khoảng cách thì trái trước.
* **`win_block_center`:** nước **thắng ngay** trước, rồi nước **chặn** đối thủ sắp thắng, cuối cùng là cột giữa. Nhược điểm: phải thử trước từng nước (qua `result` và `utility`) nên **mỗi nút tốn thời gian hơn**.

Alpha-Beta chỉ nhanh hơn khi nước tốt được xét *trước*, nên phải xem **cả số nút lẫn thời gian**.

**Nhận xét.**

* **Thế cờ đã có quân: sắp xếp giảm số nút rất mạnh.** 4×5 sau `[2,2,1,3]`: 393,369 (không sắp xếp) → 111,134 (`center_first`) → 38,678 (`win_block_center`). 5×5 sau `[2,2,2,1,3,3]`: không sắp xếp chưa xong sau 1,000,000 nút; `center_first` 864,211; `win_block_center` 162,982. Cả ba luôn cùng giá trị khi xong (có `assert`).
* **Bàn trống 4×4 thì ngược lại:** `center_first` tới 155,968 nút so với 62,889 khi không sắp xếp, `win_block_center` 67,921. Có thể vì bàn 4 cột thì cột giữa không thật sự mạnh hơn (suy đoán, chưa kiểm riêng).
* **Ít nút chưa chắc là nhanh hơn.** Mỗi nút của `win_block_center` tốn khoảng 90–290 µs, còn hai cách kia chỉ khoảng 8–9 µs (chậm hơn khoảng 10–30 lần). Kết quả: 4×4 trống, không sắp xếp nhanh nhất (0.5 giây, so với 1.3 giây của `center_first` và 6.3 giây của `win_block_center`); 4×5, `center_first` nhanh nhất (0.9 giây, so với 3.2 giây và 4.7 giây); 5×5, `center_first` xong trong 7.1 giây, `win_block_center` 25.2 giây, còn không sắp xếp chưa xong.
* **Chọn thứ tự.** Không thứ tự nào thắng ở mọi nơi. `win_block_center` ít nút nhất ở hầu hết thế cờ nhưng không nhanh nhất ở thế cờ nào, nên chỉ dùng khi cần "thứ tự ít nút nhất" (`BEST_ORDER`); các mục Loạt 1, Loạt 2 và Playtime dùng `no_ordering` vì rẻ nhất mỗi nút.

In [11]:
# Bộ thế cờ để đo (tạo bằng board_from_moves) và ba cách sắp xếp nước đi. Giới hạn 1000000 nút.
ORDER_LIMIT = 1000000
b6x7, p6x7, m6x7, seed6x7 = random_position((6, 7), 14, 20261109, quiet = True)
ordering_boards = {
    "4x4 trống":                  board_from_moves((4, 4), []),
    "4x5 sau [2,2,1,3]":          board_from_moves((4, 5), [2, 2, 1, 3]),
    "5x5 sau [2,2,2,1,3,3]":      board_from_moves((5, 5), [2, 2, 2, 1, 3, 3]),
    "4x5 bàn 5 (thắng sau 2 lượt)": boards["5. Thắng sau 2 lượt (2 đường dọa)"],
    f"6x7 còn 14 ô trống (seed {seed6x7})": (b6x7, p6x7),
}
orders = {"không sắp xếp": no_ordering, "center_first": center_first, "win_block_center": win_block_center}

rows = []
for name, (board, player) in ordering_boards.items():
    for order_name, order in orders.items():
        r = timed_search(alpha_beta_search, board, player, move_order = order, node_limit = ORDER_LIMIT)
        rows.append({"Thế cờ": name, "Cách sắp xếp": order_name,
                     "Nước": r["move"] if r["finished"] else "—", "Giá trị": r["value"] if r["finished"] else "—",
                     "Số nút": r["nodes"] if r["finished"] else f">{ORDER_LIMIT} (không xong)",
                     "Thời gian (s)": round(r["time_ms"] / 1000, 2),
                     "Thời gian cho mỗi nút (µs)": round(1000 * r["time_ms"] / r["nodes"])})

table_order = pd.DataFrame(rows)
display(table_order)

# mọi cách sắp xếp phải cho cùng giá trị trên mỗi thế cờ (khi đã xong)
for name, group in table_order.groupby("Thế cờ", sort = False):
    assert group[group["Giá trị"] != "—"]["Giá trị"].nunique() == 1, name

,Thế cờ,Cách sắp xếp,Nước,Giá trị,Số nút,Thời gian (s),Thời gian cho mỗi nút (µs)
0,4x4 trống,không sắp xếp,0,0,62889,0.56,9
1,4x4 trống,center_first,1,0,155968,1.31,8
2,4x4 trống,win_block_center,1,0,67921,7.69,113
3,"4x5 sau [2,2,1,3]",không sắp xếp,0,-1,393369,3.20,8
4,"4x5 sau [2,2,1,3]",center_first,2,-1,111134,0.91,8
5,"4x5 sau [2,2,1,3]",win_block_center,2,-1,38678,5.78,149
6,"5x5 sau [2,2,2,1,3,3]",không sắp xếp,—,—,>1000000 (không xong),8.19,8
7,"5x5 sau [2,2,2,1,3,3]",center_first,2,-1,864211,7.18,8
8,"5x5 sau [2,2,2,1,3,3]",win_block_center,2,-1,162982,30.54,187
9,4x5 bàn 5 (thắng sau 2 lượt),không sắp xếp,1,1,578,0.00,8


### The first few moves [5 points]

Start with an empty board. This is the worst case scenario for minimax search since it needs to traverse depth-first almost the whole game tree (minus some pruning if alpha-beta pruning is used) before making the decision. Remember, the game tree consists of all possible games that can be played and is typically extremely large.

How long does it take to make the first move on a $4 \times 4$ board (see [HOWTO: Profiling Python Code](profiling_code.ipynb))? Describe what your agent could do to improve this time.

**Nhận xét.**

* Nước đầu tiên trên bàn trống 4×4: không sắp xếp xét 62,889 nút, mất khoảng **0.6 giây** (`%time`); `win_block_center` xét 67,921 nút nhưng mất khoảng **7 giây** (chậm hơn khoảng 11 lần vì mỗi nút tốn công hơn).
* `%prun` (chạy dưới profiler nên tổng cộng 1.5 giây): thời gian chủ yếu ở `Position.drop` (thả quân, tạo thế cờ mới), khoảng 0.93 giây tức khoảng **60%**, gọi 62,888 lần (một lần mỗi nút); trong đó phần kiểm tra thắng quanh quân vừa đặt (`_completes_four`) chỉ khoảng 0.13 giây. Tìm các cột còn trống (`legal_columns`) khoảng 0.29 giây, tức khoảng 19%.

**Agent có thể làm gì để nhanh hơn?**

1. **Sắp xếp nước đi tốt hơn** (đã làm): giúp nhiều ở bàn có quân, không giúp ở bàn trống 4×4 (xem mục Move ordering).
2. **Nhớ các thế cờ đã tính:** `AlphaBetaAgent` đã nhớ nước ở gốc (xem Playtime); có thể nhớ cả thế cờ bên trong cây, vì một thế cờ đạt được bằng nhiều thứ tự nước.
3. **Dùng đối xứng trái / phải:** bàn lật trái-phải cho cùng kết quả, nên nước đầu tiên chỉ cần xét một nửa số cột.
4. **Bảng nước khai cuộc:** tính một lần cho vài nước đầu rồi lưu lại.
5. **Đã làm: `Position`** (trong `connect4_core.py`) kiểm tra bàn một lần rồi mỗi nước chỉ xét các đường đi qua quân vừa đặt; nước đầu tiên trên bàn trống 4×4 nhờ vậy giảm từ khoảng 6.7 giây (bản cũ) xuống khoảng 0.6 giây.
6. **Cắt độ sâu và dùng hàm đánh giá** thay vì tìm tới cuối ván: đây là Task 4 (phần của Tiến).

In [12]:
# Nước đi đầu tiên trên bàn trống 4x4, không sắp xếp. (verbose = 1 để in số nút như notebook Tic-Tac-Toe)
board44 = empty_board((4, 4))
%time info = alpha_beta_search(board44, 1, verbose=1)
print(info)


# Cùng thế cờ, thứ tự ít nút nhất (win_block_center)
%time info = alpha_beta_search(board44, 1, move_order = BEST_ORDER)
print(info)


# Xem thời gian tốn vào đâu (10 hàm tốn nhiều thời gian nhất), như profiling_code.ipynb.
# -q -r: không mở cửa sổ pager mà trả về kết quả để in ra, nếu không output sẽ không được lưu vào notebook khi chạy bằng nbconvert.
stats = %prun -l 10 -q -r alpha_beta_search(board44, 1)
stats.sort_stats("tottime").print_stats(10)

Number of nodes searched: 62889
CPU times: user 511 ms, sys: 1e+03 ns, total: 511 ms
Wall time: 511 ms
{'move': 0, 'value': 0, 'nodes': 62889}


CPU times: user 7.7 s, sys: 1.99 ms, total: 7.7 s
Wall time: 7.7 s
{'move': 1, 'value': 0, 'nodes': 67921}


          3269958 function calls (3207070 primitive calls) in 1.299 seconds

   Ordered by: internal time
   List reduced from 52 to 10 due to restriction <10>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
    62888    0.232    0.000    0.816    0.000 connect4_core.py:121(drop)
    62888    0.196    0.000    0.196    0.000 connect4_core.py:64(_completes_four)
  62889/1    0.115    0.000    1.299    1.299 connect4_minimax.py:74(visit)
   106715    0.102    0.000    0.307    0.000 numeric.py:679(flatnonzero)
    43827    0.079    0.000    0.220    0.000 connect4_core.py:106(legal_columns)
   484046    0.068    0.000    0.220    0.000 {built-in method builtins.isinstance}
   106715    0.054    0.000    0.085    0.000 fromnumeric.py:1879(ravel)
   188665    0.054    0.000    0.124    0.000 {built-in method _abc._abc_instancecheck}
   125777    0.044    0.000    0.189    0.000 connect4_core.py:37(_validate_player)
   188665    0.043    0.000    0.043    0.000 {bui

### Playtime [5 points]

Let the Minimax Search agent play a random agent on a $4 \times 4$ board. Analyze wins, losses and draws.

Mình dùng **Alpha-Beta** (không dùng Minimax thuần, vì thuần trên bàn trống 4×4 đã vượt 500,000 nút mà chưa xong, xem Loạt 1) với `AlphaBetaAgent(remember_moves = True)` và `no_ordering` (nhanh nhất trên bàn 4×4 gần trống). Mỗi lượt đi (trước / sau) đấu 100 ván, seed `20261109`; có `assert` Alpha-Beta **không thua ván nào**.

**Nhận xét.**

* **Kết quả:** đi trước 75 thắng, 25 hòa, **0 thua**; đi sau 91 thắng, 9 hòa, **0 thua**.
* **Vì sao không thua:** Alpha-Beta luôn chọn nước tối ưu, và trên bàn trống 4×4 nếu cả hai chơi đúng thì hòa (giá trị bàn trống là 0, xem bảng Move ordering), nên ít nhất cũng được hòa.
* **Vì sao vẫn có hòa:** Random chỉ cần vô tình đi đúng các nước quan trọng (như chặn đúng lúc) là Alpha-Beta không thắng được. Số ván hòa khác nhau giữa hai lượt (25 và 9) đến từ một mẫu 100 ván mỗi lượt; mình không kết luận đi trước hay đi sau tốt hơn.
* **Thời gian giảm nhờ nhớ thế cờ:** trung bình khoảng 2.7 ms mỗi nước (đi trước) và 5.9 ms (đi sau). Chia số lần gọi thành bốn phần theo thứ tự thì đi trước là 9.5 → 1.0 → 0.2 → 0.3 ms, đi sau là 20.3 → 1.9 → 0.9 → 0.7 ms: các thế cờ đầu ván gặp lại nhiều lần nên sau khi đã tính (213 thế cờ khi đi trước, 387 khi đi sau) chỉ cần tra lại. Tổng cho 100 ván khoảng 1.7 giây và 4.3 giây, nên không cần giảm số ván.

In [13]:
# Alpha-Beta (có nhớ thế cờ) đấu Random trên bàn 4x4. Mỗi lượt đi 100 ván.
PLAY_N = 100
PLAY_ORDER = no_ordering


class TimedAct:
    """bọc agent.act để ghi thời gian của MỌI lần gọi (kể cả lần lấy lại nước đã nhớ)."""
    def __init__(self, agent):
        self.agent, self.times = agent, []

    def __call__(self, board, player = 1):
        start = time.perf_counter()
        move = self.agent.act(board, player)
        self.times.append(time.perf_counter() - start)
        return move


play_rows, play_times = [], {}
for label, alpha_beta_first in [("đi trước (người chơi 1)", True), ("đi sau (người chơi -1)", False)]:
    agent = AlphaBetaAgent(move_order = PLAY_ORDER, remember_moves = True)
    timed = TimedAct(agent)
    np.random.seed(20261109)
    start = time.perf_counter()
    if alpha_beta_first:
        r = play(timed, random_player, N = PLAY_N, shape = (4, 4))
        wins, losses = r["first"], r["second"]
    else:
        r = play(random_player, timed, N = PLAY_N, shape = (4, 4))
        wins, losses = r["second"], r["first"]
    assert losses == 0, "Alpha-Beta không được thua ván nào!"
    play_times[label] = np.array(timed.times)
    play_rows.append({"Alpha-Beta đi": label, "Số ván": PLAY_N, "Thắng": wins, "Hòa": r["draw"], "Thua": losses,
                      "Thời gian TB/nước đi (ms)": round(1000 * np.mean(timed.times), 2),
                      "Số nước đã tìm thật": len(agent.nodes), "Số thế cờ đã nhớ": len(agent.known_moves),
                      "Tổng thời gian (s)": round(time.perf_counter() - start, 1)})

table_play = pd.DataFrame(play_rows)
display(table_play)

# thời gian trung bình của mỗi phần tư số lần gọi (theo thứ tự), để thấy agent nhanh dần khi đã nhớ nhiều
for label, times in play_times.items():
    parts = [round(float(1000 * p.mean()), 2) for p in np.array_split(times, 4)]
    print(f"{label}: thời gian TB/nước (ms) theo từng phần tư số lần gọi: {parts}")

fig, ax = plt.subplots(figsize = (5.5, 3.5))
x = np.arange(len(table_play))
for k, (col, color) in enumerate([("Thắng", "tab:green"), ("Hòa", "tab:gray"), ("Thua", "tab:red")]):
    ax.bar(x + (k - 1) * 0.25, table_play[col], width = 0.25, label = col, color = color)
ax.set_xticks(x)
ax.set_xticklabels(table_play["Alpha-Beta đi"])
ax.set_ylabel("Số ván")
ax.set_title("Alpha-Beta vs. Random trên bàn 4x4")
ax.legend()
plt.show()

,Alpha-Beta đi,Số ván,Thắng,Hòa,Thua,Thời gian TB/nước đi (ms),Số nước đã tìm thật,Số thế cờ đã nhớ,Tổng thời gian (s)
0,đi trước (người chơi 1),100,75,25,0,2.49,213,213,1.6
1,đi sau (người chơi -1),100,91,9,0,5.27,387,387,3.8


đi trước (người chơi 1): thời gian TB/nước (ms) theo từng phần tư số lần gọi: [8.68, 0.82, 0.18, 0.25]
đi sau (người chơi -1): thời gian TB/nước (ms) theo từng phần tư số lần gọi: [17.93, 1.79, 0.72, 0.6]


## Task 4: Heuristic Alpha-Beta Tree Search

### Heuristic evaluation function [10 points]

The evaluator scores each uncontested four-cell window: one, two, or three aligned discs receive increasing weights, while the opponent's patterns subtract from the score. A small center-column bonus breaks ties. Terminal outcomes are returned exactly as +1, -1, or 0; `tanh` bounds every non-terminal score to (-1, 1).

In [14]:
# Algorithm source from Tiến's PR #59; integrated without replacing Task 5.
from connect4_heuristic import heuristic_evaluation
print(inspect.getsource(heuristic_evaluation))

def heuristic_evaluation(board, player=1):
    """Score a board from player's perspective, always returning a value in [-1, 1]."""
    outcome = utility(board, player)
    if outcome is not None:
        return float(outcome)

    rows, columns = board.shape
    weights = {1: 1.0, 2: 6.0, 3: 30.0}
    score = 0.0

    for row in range(rows):
        for column in range(columns):
            for row_step, column_step in _DIRECTIONS:
                end_row = row + (_WIN_LENGTH - 1) * row_step
                end_column = column + (_WIN_LENGTH - 1) * column_step
                if not (0 <= end_row < rows and 0 <= end_column < columns):
                    continue

                window = [int(board[row + offset * row_step,
                                    column + offset * column_step])
                          for offset in range(_WIN_LENGTH)]
                own_count = window.count(player)
                opponent_count = window.count(-player)
                if opponent_count 

### Cutting Off Search [10 points]
 
The agent searches a configurable total number of plies, including each candidate move. Terminal outcomes are checked before cutoff evaluation; remaining leaves use the heuristic. Alpha-beta bounds prune branches that cannot change the decision, and heuristic move ordering tries promising children first. Increasing depth can improve tactical foresight, but increases search work.

In [15]:
from connect4_heuristic import HeuristicAlphaBetaAgent
print(inspect.getsource(HeuristicAlphaBetaAgent))

class HeuristicAlphaBetaAgent:
    """Depth-limited alpha-beta agent; call as agent(board, player)."""

    def __init__(self, depth=3):
        if isinstance(depth, bool) or not isinstance(depth, Integral) or depth < 1:
            raise ValueError("depth must be a positive integer")
        self.depth = int(depth)
        self.nodes = 0
        self.last_elapsed = 0.0

    def __call__(self, board, player=1):
        utility(board, player)  # Validate the board and player before searching.
        legal_actions = actions(board)
        if not legal_actions:
            raise ValueError("agent cannot move from a terminal state")

        self.nodes = 1
        started = perf_counter()
        center = (board.shape[1] - 1) / 2

        def ordered_children(state, to_move):
            ordered_columns = sorted(actions(state), key=lambda col: (abs(col - center), col))
            children = []
            for column in ordered_columns:
                child = result(state, to_move, colum

The smoke tests use a standard $6 \times 7$ board and cover the opening, horizontal/vertical/diagonal wins, and an immediate block. They assert the chosen columns as well as the evaluator's bounds and exact terminal score.

In [16]:
# Five legal positions cover the opening, immediate wins, blocking, and a diagonal threat.
empty = np.zeros((6, 7), dtype=int)
horizontal_win = empty.copy()
horizontal_win[-1, :3] = 1
vertical_win = empty.copy()
vertical_win[-3:, 2] = -1
blocking_position = np.zeros((6, 7), dtype=int)
blocking_position[-1, :3] = -1

diagonal_win = np.zeros((6, 7), dtype=int)
diagonal_win[-1, 0] = 1
diagonal_win[-1, 1] = -1
diagonal_win[-2, 1] = 1
diagonal_win[-1, 2] = -1
diagonal_win[-2, 2] = -1
diagonal_win[-3, 2] = 1
diagonal_win[-1, 3] = -1
diagonal_win[-2, 3] = -1
diagonal_win[-3, 3] = -1

agent = HeuristicAlphaBetaAgent(depth=3)
assert agent(empty, 1) == 3
assert agent(horizontal_win, 1) == 3
assert agent(vertical_win, -1) == 2
assert agent(blocking_position, 1) == 3
assert agent(diagonal_win, 1) == 3

scored_positions = (empty, horizontal_win, vertical_win, blocking_position, diagonal_win)
scores = [heuristic_evaluation(position, side)
          for position, side in zip(scored_positions, (1, 1, -1, 1, 1))]
assert all(-1.0 <= score <= 1.0 for score in scores)
assert heuristic_evaluation(result(horizontal_win, 1, 3), 1) == 1.0
print("Five tactical checks passed; heuristic scores:", [round(score, 4) for score in scores])

Five tactical checks passed; heuristic scores: [0.0, 0.3307, 0.4137, -0.3307, -0.6455]


The table measures empty-board first moves while increasing columns from four to seven, then increasing rows to the standard $6 \times 7$ size. It reports both recursive search nodes and elapsed time for cutoffs 1-3. Node counts are more portable than timings; growth with depth and board size illustrates why shallow cutoff search is used instead of exhaustive minimax.

In [17]:
experiment_results = []
for shape in ((4, 4), (4, 5), (4, 6), (4, 7), (5, 7), (6, 7)):
    position = np.zeros(shape, dtype=int)
    for depth in (1, 2, 3):
        timed_agent = HeuristicAlphaBetaAgent(depth=depth)
        move = timed_agent(position, 1)
        experiment_results.append((shape, depth, timed_agent.nodes,
                                   timed_agent.last_elapsed, move))

print(f"{'Board':<8} {'Depth':>5} {'Nodes':>8} {'Seconds':>10} {'Move':>6}")
for shape, depth, nodes, elapsed, move in experiment_results:
    print(f"{shape[0]}x{shape[1]:<6} {depth:>5} {nodes:>8} {elapsed:>10.6f} {move:>6}")

Board    Depth    Nodes    Seconds   Move
4x4          1        5   0.000598      0
4x4          2       12   0.001889      0
4x4          3       31   0.004866      0
4x5          1        6   0.000799      1
4x5          2       15   0.003345      1
4x5          3       60   0.011596      2
4x6          1        7   0.001282      2
4x6          2       18   0.005212      2
4x6          3       71   0.016354      2
4x7          1        8   0.001506      3
4x7          2       21   0.007770      3
4x7          3       82   0.023976      3
5x7          1        8   0.001824      3
5x7          2       21   0.009395      3
5x7          3       92   0.031181      3
6x7          1        8   0.002111      3
6x7          2       21   0.010748      3
6x7          3       92   0.036341      3


### Playtime [5 points]

One deterministic match compares depth 2 and depth 3 on a $4 \times 5$ board. The result reports the winner by side (`1` is the first/depth-2 agent, `-1` the second/depth-3 agent) and the final board. A single match is only an example: starting player and cutoff depth both affect the outcome, so it is not evidence that one depth is generally stronger.

In [18]:
from connect4_heuristic import play_heuristic_match

match_result = play_heuristic_match(HeuristicAlphaBetaAgent(depth=2),
                                    HeuristicAlphaBetaAgent(depth=3))
print(f"Depth-2 vs depth-3 on 4x5: winner={match_result['winner']}, "
      f"plies={match_result['plies']}")
print("Winner 1 is the depth-2 agent; winner -1 is the depth-3 agent; None is a draw.")
print(match_result["board"])

Depth-2 vs depth-3 on 4x5: winner=1, plies=7
Winner 1 is the depth-2 agent; winner -1 is the depth-3 agent; None is a draw.
[[ 0  0  0  0  0]
 [ 0  0  0  0  0]
 [ 0 -1 -1 -1  0]
 [ 1  1  1  1  0]]


## Task 5: Pure Monte Carlo Search [15 points]

Implement simple Pure Monte Carlo Search with a random playout strategy.

In [19]:
# Task 5 (Toàn Thắng): chỉ import thêm những gì Task 1 - 4 chưa import. Mã nằm trong connect4_montecarlo.py.
from collections import Counter

from connect4_montecarlo import PMCSAgent, playout, playouts, pmcs, pmcs_values

SEED = 20261110    # seed gốc của Thắng (Tú dùng 20261108, Thiện dùng 20261109)

Cách làm (ý tưởng lấy từ `tictactoe_pure_monte_carlo_search.ipynb`):

1. Với **mỗi cột** còn thả được quân, đánh thử `n` ván ngẫu nhiên tới hết (hai bên đều thả quân vào cột còn trống bất kỳ). Tổng `N` ván được chia đều: `n = N // số cột`.
2. Mỗi ván cho điểm `+1` (mình thắng), `0` (hòa), `-1` (mình thua). Tính điểm trung bình của từng cột.
3. Chọn cột có điểm trung bình cao nhất (bằng điểm thì lấy cột bên trái, giống notebook Tic-Tac-Toe).

Không cần tự nghĩ ra hàm chấm điểm thế cờ, chỉ cần biết luật để đánh thử tới hết ván.

Chỗ khác với Tic-Tac-Toe:

* Người chơi là `1` / `-1` (đối thủ của `player` là `-player`), game lấy từ `connect4_core.py`.
* **Ván đánh thử dùng `Position`** (`Position.from_board` một lần, rồi `drop` / `legal_columns` / `utility`) thay cho `random_player` + `result` + `utility`. Hai cách chọn nước **giống hệt nhau** (chọn đều trong các cột còn trống), nhưng `result` / `utility` kiểm tra lại cả bàn ở mỗi nước nên chậm hơn nhiều. Thiện cũng làm vậy ở `connect4_minimax.py`.
* **`n` ít nhất là 1** (`n = max(1, N // số cột)`). Công thức gốc cho `n = 0` nếu `N` nhỏ hơn số cột, khi đó không có ván nào để lấy trung bình.
* Tác tử là một **class** `PMCSAgent` với `act(board, player = 1)` (đúng chữ ký của `random_player`), lưu thời gian mỗi nước (`move_times`) theo `store_agent_state_information.ipynb`.

In [20]:
# Xem lại mã nguồn đã cài trong connect4_montecarlo.py (chỉ in ra để đọc)
for f in (playout, pmcs_values, pmcs, PMCSAgent):
    print(inspect.getsource(f))

def playout(state, action, player = 1):
    """Perform a random playout starting with the given action on the given board
    and return the utility of the finished game (+1 win, 0 draw, -1 loss for player).

    The input board is not changed. The random policy picks uniformly among the columns
    that are not full, which is what random_player does too; Position is used because
    random_player / result check the whole board again at every move and are much slower.
    """
    return _playout_from_position(Position.from_board(state), action, player)

def pmcs_values(board, N = 100, player = 1, verbose = 0):
    """Average utility of every legal action. The N playouts are evenly divided between the actions.

    N is a requested *approximate* total, not always the executed total. We run
    max(1, N // number_of_actions) playouts per action, uniformly. The actual count
    is number_of_actions * max(1, N // number_of_actions), so e.g. N=10 gives 7
    trials on an empty 6x7 board. Pr

In [21]:
# Minh họa: bàn trống 6x7, N = 700 (100 ván đánh thử cho mỗi cột), in điểm trung bình từng cột (verbose = 1).
board = empty_board((6, 7))
np.random.seed(SEED)
%time move = pmcs(board, N = 700, verbose = 1)
print("Cột được chọn:", move)

# Thời gian của một ván đánh thử và của một nước đi trên bàn chuẩn 6x7
np.random.seed(SEED)
start = time.perf_counter()
playouts(board, 3, 1, N = 2000)
playout_ms = 1000 * (time.perf_counter() - start) / 2000
np.random.seed(SEED)
start = time.perf_counter()
pmcs(board, N = 1000)
print(f"Một ván đánh thử trên bàn 6x7: {playout_ms:.3f} ms | một nước đi với N = 1000: {1000 * (time.perf_counter() - start):.0f} ms")

Actions: [0, 1, 2, 3, 4, 5, 6] (700 requested; 700 actual playouts = 100 per action)
{0: -0.2, 1: -0.06, 2: 0.19, 3: 0.34, 4: 0.33, 5: -0.06, 6: 0.02}
CPU times: user 278 ms, sys: 70.1 ms, total: 349 ms
Wall time: 289 ms
Cột được chọn: 3


Một ván đánh thử trên bàn 6x7: 0.406 ms | một nước đi với N = 1000: 357 ms

**Nhận xét.** Mã được in ở ô trên có cấu trúc giống `playout` / `playouts` / `pmcs` của notebook Tic-Tac-Toe: `playout` thả quân đầu tiên rồi cho hai bên đi bừa tới hết ván, `pmcs_values` tính điểm trung bình của từng cột, `pmcs` lấy cột có điểm cao nhất. Ở ô minh họa, bàn trống 6×7 với N = 700 (100 ván đánh thử mỗi cột) cho điểm trung bình `{0: -0.2, 1: -0.06, 2: 0.19, 3: 0.34, 4: 0.33, 5: -0.06, 6: 0.02}` và chọn **cột 3 (cột giữa)**. Cột giữa và cột sát nó (cột 4) có điểm cao nhất, hai cột ngoài rìa (cột 0 và 6) thấp nhất. Lý do: cột giữa nằm trong nhiều đường bốn quân nhất, cùng lý do `center_first` giúp Alpha-Beta cắt tỉa ở Task 3. Lần chạy này cột 4 chỉ kém cột 3 có 0.01 và mỗi cột chỉ 100 ván, nên chênh lệch đó nằm trong mức dao động, không kết luận cột 3 hơn cột 4 được.

**Tốc độ.** Một ván đánh thử trên bàn 6×7 mất khoảng **0.3 - 0.4 ms**, và một nước đi với N = 1,000 mất khoảng **0.3 - 0.4 giây** (mỗi lần chạy lệch khoảng 20 - 30%, đo trên máy chạy notebook này). Nghĩa là Pure Monte Carlo **chạy được ngay trên bàn chuẩn 6×7**, điều mà Alpha-Beta đầy đủ ở Task 3 không làm được: Loạt 2 của Thiện cho thấy Alpha-Beta đã vượt 500,000 nút khi chỉ còn 20 ô trống. Lý do là thời gian của Pure Monte Carlo chỉ phụ thuộc vào `N` và độ dài ván đánh thử (tối đa 42 nước), không bùng nổ theo độ sâu như cây tìm kiếm đầy đủ. Dùng `Position` trong ván đánh thử là điều giúp nhanh như vậy: `random_player` và `result` kiểm tra lại cả bàn ở mỗi nước nên chậm hơn nhiều.

Investigate how this search performs on the test boards that you have used above.

Dùng **đúng sáu bàn của Task 3**. Với mỗi bàn và mỗi `N` trong `{10, 100, 1000}`: gọi `pmcs` **20 lần** (lần `r` dùng seed `20261110 + r`), ghi số lần chọn **đúng**, nước hay chọn nhất và thời gian trung bình mỗi nước. "Đúng" dùng **cùng tiêu chí với cột "Đúng?" ở Task 3**: nước nằm trong `expected_moves`, hoặc nước đó vẫn giữ được thắng (`keeps_the_win`).

In [22]:
N_LIST = [10, 100, 1000]
REPEATS = 20

keep_cache = {}
def is_correct(board, player, move, expected):
    """cùng tiêu chí "Đúng?" của Task 3: nước mong đợi, hoặc nước vẫn giữ được thắng."""
    if move in expected: return True
    key = (board.tobytes(), board.shape, player, move)
    if key not in keep_cache: keep_cache[key] = bool(keeps_the_win(board, player, move))
    return keep_cache[key]


rows = []
for name, (board, player) in boards.items():
    expected = expected_moves(board, player)
    for N in N_LIST:
        chosen, total = [], 0.0
        for r in range(REPEATS):
            np.random.seed(SEED + r)
            start = time.perf_counter()
            chosen.append(pmcs(board, N = N, player = player))
            total += time.perf_counter() - start
        counts = Counter(chosen)
        rows.append({"Bàn": name, "Kích thước": f"{board.shape[0]}x{board.shape[1]}", "Ô trống": int(np.sum(board == 0)),
                     "Nước mong đợi": expected, "N": N,
                     "Số lần đúng /20": sum(is_correct(board, player, m, expected) * c for m, c in counts.items()),
                     "Nước hay chọn nhất": counts.most_common(1)[0][0],
                     "Các nước đã chọn (số lần)": str(dict(counts.most_common())),
                     "Thời gian TB/nước (ms)": round(1000 * total / REPEATS, 1)})

table_boards_mc = pd.DataFrame(rows)
display(table_boards_mc)

,Bàn,Kích thước,Ô trống,Nước mong đợi,N,Số lần đúng /20,Nước hay chọn nhất,Các nước đã chọn (số lần),Thời gian TB/nước (ms)
0,1. Thắng ngay (hàng ngang),4x4,10,[3],10,14,3,"{3: 14, 2: 3, 1: 3}",0.8
1,1. Thắng ngay (hàng ngang),4x4,10,[3],100,20,3,{3: 20},6.3
2,1. Thắng ngay (hàng ngang),4x4,10,[3],1000,20,3,{3: 20},67.1
3,2. Thắng ngay (hàng dọc),4x4,10,[0],10,20,0,{0: 20},0.7
4,2. Thắng ngay (hàng dọc),4x4,10,[0],100,20,0,{0: 20},7.6
5,2. Thắng ngay (hàng dọc),4x4,10,[0],1000,20,0,{0: 20},79.3
6,3. Thắng ngay (đường chéo),4x4,6,[3],10,9,0,"{0: 9, 3: 9, 1: 2}",0.5
7,3. Thắng ngay (đường chéo),4x4,6,[3],100,20,3,{3: 20},4.9
8,3. Thắng ngay (đường chéo),4x4,6,[3],1000,20,3,{3: 20},54.3
9,4. Phải chặn đối thủ,4x4,11,[3],10,7,3,"{3: 7, 1: 6, 0: 5, 2: 2}",1.0


### Đặt cạnh kết quả Alpha-Beta của Thiện

Cùng sáu bàn, Alpha-Beta thứ tự mặc định (`no_ordering`) của Thiện (Task 3, bảng "Alpha-Beta (mặc định)"). Mình **gọi lại đúng hàm của Thiện** (`timed_search(alpha_beta_search, ...)`, không viết lại) rồi `assert` số nút khớp với số trong bảng Task 3 (812, 162, 154, 4,733, 578, 38). Số nút là con số cố định; thời gian thì đo trên máy chạy notebook này.

In [23]:
task3_nodes = [812, 162, 154, 4733, 578, 38]   # "Alpha-Beta (mặc định)" trong bảng Task 3 của Thiện

rows = []
for (name, (board, player)), nodes_task3 in zip(boards.items(), task3_nodes):
    expected = expected_moves(board, player)
    r = timed_search(alpha_beta_search, board, player, node_limit = 300000, move_order = no_ordering)
    assert r["finished"] and r["nodes"] == nodes_task3, f"{name}: {r['nodes']} nút, bảng Task 3 ghi {nodes_task3}"
    row = {"Bàn": name, "Nước mong đợi": expected,
           "Alpha-Beta: số nút": r["nodes"], "Alpha-Beta: thời gian (ms)": round(r["time_ms"], 1),
           "Alpha-Beta: đúng?": r["move"] in expected or (r["value"] == 1 and keeps_the_win(board, player, r["move"]))}
    for N in N_LIST:
        sub = table_boards_mc[(table_boards_mc["Bàn"] == name) & (table_boards_mc["N"] == N)].iloc[0]
        row[f"Monte Carlo N={N}: đúng /20"] = sub["Số lần đúng /20"]
        row[f"Monte Carlo N={N}: ms"] = sub["Thời gian TB/nước (ms)"]
    rows.append(row)

table_vs_ab = pd.DataFrame(rows)
display(table_vs_ab)
assert table_vs_ab["Alpha-Beta: đúng?"].all()

,Bàn,Nước mong đợi,Alpha-Beta: số nút,Alpha-Beta: thời gian (ms),Alpha-Beta: đúng?,Monte Carlo N=10: đúng /20,Monte Carlo N=10: ms,Monte Carlo N=100: đúng /20,Monte Carlo N=100: ms,Monte Carlo N=1000: đúng /20,Monte Carlo N=1000: ms
0,1. Thắng ngay (hàng ngang),[3],812,11.3,True,14,0.8,20,6.3,20,67.1
1,2. Thắng ngay (hàng dọc),[0],162,1.8,True,20,0.7,20,7.6,20,79.3
2,3. Thắng ngay (đường chéo),[3],154,1.6,True,9,0.5,20,4.9,20,54.3
3,4. Phải chặn đối thủ,[3],4733,40.4,True,7,1.0,20,9.8,20,99.6
4,5. Thắng sau 2 lượt (2 đường dọa),[1],578,4.7,True,10,1.0,19,9.4,20,85.5
5,6. Bàn chuẩn 6x7 cuối ván,[3],38,0.4,True,15,0.5,20,3.6,20,34.7


**Nhận xét.**

* **Với N ≥ 100, Monte Carlo gần như luôn tìm ra nước thắng ngay / nước phải chặn.** Ở N = 100, 5 bàn trên 6 đúng 20/20 lần, riêng bàn 5 "thắng sau 2 lượt" đúng 19/20 (một lần chọn cột 3 thay vì cột 1). Ở N = 1,000 **cả 6 bàn đều đúng 20/20**.
* **N = 10 thì may rủi**, vì các bàn nhỏ ở đây (4 hoặc 5 cột) chỉ được `10 // 4 = 2` hoặc `10 // 5 = 2` ván cho mỗi cột (bàn 6×7 có 7 cột: 1 ván mỗi cột). Số lần đúng /20: bàn 1: 14, bàn 2: 20, bàn 3: 9, bàn 4: 7, bàn 5: 10, bàn 6: 15. Bàn 2 đúng 20/20 là do **may**: cột thắng ngay là cột 0, ván thử bắt đầu từ nước thắng nên luôn ra `+1`, và khi nhiều cột bằng điểm máy lấy cột bên trái (cột 0), nên cột 0 không bao giờ bị mất. Bàn 4 "phải chặn" thấp nhất (7/20), cột đúng chỉ được chọn 7 lần trong khi `{3: 7, 1: 6, 0: 5, 2: 2}`: ván thử cho đối thủ đi bừa, nên cột chặn không hơn hẳn các cột khác khi chỉ có 2 ván thử.
* **Bàn 5 "thắng sau 2 lượt" khó hơn** các bàn thắng ngay vì phải nhìn xa hơn một nước (tạo hai đường dọa thắng cùng lúc): đây là bàn duy nhất N = 100 chưa đạt 20/20 (19/20), và N = 10 chỉ đúng 10/20 lần.
* **Alpha-Beta đúng 100% ở cả 6 bàn, nhưng tìm hết:** số nút lần lượt là 812, 162, 154, 4,733, 578 và 38 (khớp bảng Task 3 của Thiện, có `assert`). Monte Carlo cần `N` đủ lớn (≥ 100) mới chắc đúng, và ở mức đó hai bên có thời gian cùng cỡ vài ms tới vài chục ms: Alpha-Beta mất từ khoảng 0.4 ms (bàn 6, chỉ 38 nút) tới khoảng 40 ms (bàn 4, 4,733 nút), còn Monte Carlo (N = 100) mất khoảng 6 - 13 ms ở mọi bàn. Nghĩa là Alpha-Beta nhanh hơn khi cây nhỏ và chậm hơn khi cây lớn hơn. Sáu bàn này là bàn nhỏ hoặc gần hết ván, nơi Alpha-Beta tìm được nhanh. Thời gian của Monte Carlo gần như cố định theo `N` (khoảng 0.5 - 1 ms với N = 10, 6 - 13 ms với N = 100, 70 - 120 ms với N = 1,000), còn thời gian của Alpha-Beta tăng rất nhanh theo số nút. Thời gian đo trên máy chạy notebook này, mỗi lần chạy lệch khoảng 20 - 30%.

Which implementation plays stronger stronger? Discuss why you think that is the case and what it depends on.

Đề hỏi "cách nào chơi mạnh hơn". Nhóm có ba cách: Alpha-Beta đầy đủ (Task 3, Thiện), Heuristic Alpha-Beta (Task 4, Tiến) và Pure Monte Carlo (Task 5, Thắng). Mình cho Monte Carlo đấu:

1. **Alpha-Beta trên bàn 4×4**, nơi Alpha-Beta tìm tới hết ván (chơi không sai). Dùng `AlphaBetaAgent(move_order = no_ordering, remember_moves = True)` của Thiện; mỗi cấu hình **20 ván Monte Carlo đi trước + 20 ván đi sau**. Có `assert` Alpha-Beta không thua ván nào.
2. **Random trên bàn chuẩn 6×7**: N = 10 và 100 mỗi cấu hình 100 ván đi trước + 100 ván đi sau; N = 1000 chỉ 20 ván mỗi bên (cho đỡ lâu).
3. **Heuristic Alpha-Beta của Tiến trên bàn 6×7**: xem ô cuối của mục này.

Mỗi trận một ô code, đặt seed `20261110` trước mỗi trận. `play` của Thiện trả `{"first", "second", "draw"}`, nên trong bảng **Bên A là bên đi trước, Bên B là bên đi sau**.

In [24]:
match_rows = []
alpha_beta = AlphaBetaAgent(move_order = no_ordering, remember_moves = True)   # dùng chung để nhớ các thế cờ đã tính


def run_match(match, board_name, mc, opponent, opponent_name, mc_first, games, shape):
    """cho Monte Carlo (PMCSAgent `mc`) đấu `opponent` (một hàm agent(board, player)) `games` ván.
    mc_first = True: Monte Carlo đi trước. Trả về kết quả của `play` ({"first", "second", "draw"})."""
    np.random.seed(SEED)
    mc.move_times.clear()
    first, second = (mc.act, opponent) if mc_first else (opponent, mc.act)
    r = play(first, second, N = games, shape = shape)
    match_rows.append({"Trận": match, "Bàn": board_name, "Monte Carlo đi trước": mc_first,
                       "Bên A (đi trước)": mc.name if mc_first else opponent_name,
                       "Bên B (đi sau)": opponent_name if mc_first else mc.name,
                       "Số ván": games, "A thắng": r["first"], "Hòa": r["draw"], "B thắng": r["second"],
                       "Thời gian TB/nước của Monte Carlo (ms)": round(float(1000 * np.mean(mc.move_times)), 2)})
    print(match_rows[-1])
    return r

In [25]:
# Trận 1: Monte Carlo (N=100) đi TRƯỚC, Alpha-Beta đi sau, bàn 4x4, 20 ván
mc = PMCSAgent(N = 100)
r = run_match("1. Monte Carlo (100) vs Alpha-Beta", "4x4", mc, alpha_beta.act, "Alpha-Beta", True, 20, (4, 4))
assert r["second"] >= 0 and r["first"] == 0, "Monte Carlo thắng Alpha-Beta?! (Alpha-Beta chơi hoàn hảo trên 4x4 nên không được thua)"

{'Trận': '1. Monte Carlo (100) vs Alpha-Beta', 'Bàn': '4x4', 'Monte Carlo đi trước': True, 'Bên A (đi trước)': 'PMCS (100)', 'Bên B (đi sau)': 'Alpha-Beta', 'Số ván': 20, 'A thắng': 0, 'Hòa': 9, 'B thắng': 11, 'Thời gian TB/nước của Monte Carlo (ms)': 10.97}


In [26]:
# Trận 1: Alpha-Beta đi TRƯỚC, Monte Carlo (N=100) đi sau, bàn 4x4, 20 ván
mc = PMCSAgent(N = 100)
r = run_match("1. Alpha-Beta vs Monte Carlo (100)", "4x4", mc, alpha_beta.act, "Alpha-Beta", False, 20, (4, 4))
assert r["first"] >= 0 and r["second"] == 0, "Monte Carlo thắng Alpha-Beta?! (Alpha-Beta chơi hoàn hảo trên 4x4 nên không được thua)"

{'Trận': '1. Alpha-Beta vs Monte Carlo (100)', 'Bàn': '4x4', 'Monte Carlo đi trước': False, 'Bên A (đi trước)': 'Alpha-Beta', 'Bên B (đi sau)': 'PMCS (100)', 'Số ván': 20, 'A thắng': 3, 'Hòa': 17, 'B thắng': 0, 'Thời gian TB/nước của Monte Carlo (ms)': 9.66}


In [27]:
# Trận 1: Monte Carlo (N=1000) đi TRƯỚC, Alpha-Beta đi sau, bàn 4x4, 20 ván
mc = PMCSAgent(N = 1000)
r = run_match("1. Monte Carlo (1000) vs Alpha-Beta", "4x4", mc, alpha_beta.act, "Alpha-Beta", True, 20, (4, 4))
assert r["second"] >= 0 and r["first"] == 0, "Monte Carlo thắng Alpha-Beta?! (Alpha-Beta chơi hoàn hảo trên 4x4 nên không được thua)"

{'Trận': '1. Monte Carlo (1000) vs Alpha-Beta', 'Bàn': '4x4', 'Monte Carlo đi trước': True, 'Bên A (đi trước)': 'PMCS (1000)', 'Bên B (đi sau)': 'Alpha-Beta', 'Số ván': 20, 'A thắng': 0, 'Hòa': 7, 'B thắng': 13, 'Thời gian TB/nước của Monte Carlo (ms)': 114.15}


In [28]:
# Trận 1: Alpha-Beta đi TRƯỚC, Monte Carlo (N=1000) đi sau, bàn 4x4, 20 ván
mc = PMCSAgent(N = 1000)
r = run_match("1. Alpha-Beta vs Monte Carlo (1000)", "4x4", mc, alpha_beta.act, "Alpha-Beta", False, 20, (4, 4))
assert r["first"] >= 0 and r["second"] == 0, "Monte Carlo thắng Alpha-Beta?! (Alpha-Beta chơi hoàn hảo trên 4x4 nên không được thua)"

{'Trận': '1. Alpha-Beta vs Monte Carlo (1000)', 'Bàn': '4x4', 'Monte Carlo đi trước': False, 'Bên A (đi trước)': 'Alpha-Beta', 'Bên B (đi sau)': 'PMCS (1000)', 'Số ván': 20, 'A thắng': 0, 'Hòa': 20, 'B thắng': 0, 'Thời gian TB/nước của Monte Carlo (ms)': 102.0}


In [29]:
# Trận 2: Monte Carlo (N=10) đi TRƯỚC, Random đi sau, bàn 6x7, 100 ván
mc = PMCSAgent(N = 10)
run_match("2. Monte Carlo (10) vs Random", "6x7", mc, random_player, "Random", True, 100, (6, 7))

{'Trận': '2. Monte Carlo (10) vs Random', 'Bàn': '6x7', 'Monte Carlo đi trước': True, 'Bên A (đi trước)': 'PMCS (10)', 'Bên B (đi sau)': 'Random', 'Số ván': 100, 'A thắng': 81, 'Hòa': 0, 'B thắng': 19, 'Thời gian TB/nước của Monte Carlo (ms)': 2.19}


{'first': 81, 'second': 19, 'draw': 0}

In [30]:
# Trận 2: Random đi TRƯỚC, Monte Carlo (N=10) đi sau, bàn 6x7, 100 ván
mc = PMCSAgent(N = 10)
run_match("2. Random vs Monte Carlo (10)", "6x7", mc, random_player, "Random", False, 100, (6, 7))

{'Trận': '2. Random vs Monte Carlo (10)', 'Bàn': '6x7', 'Monte Carlo đi trước': False, 'Bên A (đi trước)': 'Random', 'Bên B (đi sau)': 'PMCS (10)', 'Số ván': 100, 'A thắng': 29, 'Hòa': 0, 'B thắng': 71, 'Thời gian TB/nước của Monte Carlo (ms)': 1.99}


{'first': 29, 'second': 71, 'draw': 0}

In [31]:
# Trận 2: Monte Carlo (N=100) đi TRƯỚC, Random đi sau, bàn 6x7, 100 ván
mc = PMCSAgent(N = 100)
run_match("2. Monte Carlo (100) vs Random", "6x7", mc, random_player, "Random", True, 100, (6, 7))

{'Trận': '2. Monte Carlo (100) vs Random', 'Bàn': '6x7', 'Monte Carlo đi trước': True, 'Bên A (đi trước)': 'PMCS (100)', 'Bên B (đi sau)': 'Random', 'Số ván': 100, 'A thắng': 100, 'Hòa': 0, 'B thắng': 0, 'Thời gian TB/nước của Monte Carlo (ms)': 22.21}


{'first': 100, 'second': 0, 'draw': 0}

In [32]:
# Trận 2: Random đi TRƯỚC, Monte Carlo (N=100) đi sau, bàn 6x7, 100 ván
mc = PMCSAgent(N = 100)
run_match("2. Random vs Monte Carlo (100)", "6x7", mc, random_player, "Random", False, 100, (6, 7))

{'Trận': '2. Random vs Monte Carlo (100)', 'Bàn': '6x7', 'Monte Carlo đi trước': False, 'Bên A (đi trước)': 'Random', 'Bên B (đi sau)': 'PMCS (100)', 'Số ván': 100, 'A thắng': 4, 'Hòa': 0, 'B thắng': 96, 'Thời gian TB/nước của Monte Carlo (ms)': 20.75}


{'first': 4, 'second': 96, 'draw': 0}

In [33]:
# Trận 2: Monte Carlo (N=1000) đi TRƯỚC, Random đi sau, bàn 6x7, 20 ván
mc = PMCSAgent(N = 1000)
run_match("2. Monte Carlo (1000) vs Random", "6x7", mc, random_player, "Random", True, 20, (6, 7))

{'Trận': '2. Monte Carlo (1000) vs Random', 'Bàn': '6x7', 'Monte Carlo đi trước': True, 'Bên A (đi trước)': 'PMCS (1000)', 'Bên B (đi sau)': 'Random', 'Số ván': 20, 'A thắng': 20, 'Hòa': 0, 'B thắng': 0, 'Thời gian TB/nước của Monte Carlo (ms)': 222.69}


{'first': 20, 'second': 0, 'draw': 0}

In [34]:
# Trận 2: Random đi TRƯỚC, Monte Carlo (N=1000) đi sau, bàn 6x7, 20 ván
mc = PMCSAgent(N = 1000)
run_match("2. Random vs Monte Carlo (1000)", "6x7", mc, random_player, "Random", False, 20, (6, 7))

{'Trận': '2. Random vs Monte Carlo (1000)', 'Bàn': '6x7', 'Monte Carlo đi trước': False, 'Bên A (đi trước)': 'Random', 'Bên B (đi sau)': 'PMCS (1000)', 'Số ván': 20, 'A thắng': 0, 'Hòa': 0, 'B thắng': 20, 'Thời gian TB/nước của Monte Carlo (ms)': 226.15}


{'first': 0, 'second': 20, 'draw': 0}

**Trận 3: Pure Monte Carlo so với Heuristic Alpha-Beta của Tiến (Connect 4 chuẩn 6×7).**

Sau khi tích hợp thuật toán của Tiến từ PR #59 vào `connect4_heuristic.py`, cho PMCS (N=100) đấu với cutoff 2 và 3. Mỗi cấu hình chơi 5 ván ở mỗi lượt đi; đặt lại seed để có thể tái lập, sau đó bổ sung kết quả vào bảng và biểu đồ phía dưới. Đây chỉ là một phép so sánh nhỏ, không đủ để khẳng định thuật toán nào mạnh hơn trong mọi thế cờ.

In [35]:
# Trận 3: Monte Carlo vs Heuristic Alpha-Beta của Tiến, 5 ván / phía / cutoff.
# Không đổi các kết quả Task 5 đã chạy trước đó.
for cutoff in (2, 3):
    heuristic_agent = HeuristicAlphaBetaAgent(depth=cutoff)
    for mc_first in (True, False):
        mc_agent = PMCSAgent(N=100)
        run_match(f'3. PMCS (100) vs Heuristic cutoff={cutoff}', '6x7', mc_agent,
                  heuristic_agent, f'Heuristic AB depth {cutoff}', mc_first, 5, (6, 7))


{'Trận': '3. PMCS (100) vs Heuristic cutoff=2', 'Bàn': '6x7', 'Monte Carlo đi trước': True, 'Bên A (đi trước)': 'PMCS (100)', 'Bên B (đi sau)': 'Heuristic AB depth 2', 'Số ván': 5, 'A thắng': 1, 'Hòa': 0, 'B thắng': 4, 'Thời gian TB/nước của Monte Carlo (ms)': 18.34}


{'Trận': '3. PMCS (100) vs Heuristic cutoff=2', 'Bàn': '6x7', 'Monte Carlo đi trước': False, 'Bên A (đi trước)': 'Heuristic AB depth 2', 'Bên B (đi sau)': 'PMCS (100)', 'Số ván': 5, 'A thắng': 5, 'Hòa': 0, 'B thắng': 0, 'Thời gian TB/nước của Monte Carlo (ms)': 18.24}


{'Trận': '3. PMCS (100) vs Heuristic cutoff=3', 'Bàn': '6x7', 'Monte Carlo đi trước': True, 'Bên A (đi trước)': 'PMCS (100)', 'Bên B (đi sau)': 'Heuristic AB depth 3', 'Số ván': 5, 'A thắng': 0, 'Hòa': 0, 'B thắng': 5, 'Thời gian TB/nước của Monte Carlo (ms)': 21.91}


{'Trận': '3. PMCS (100) vs Heuristic cutoff=3', 'Bàn': '6x7', 'Monte Carlo đi trước': False, 'Bên A (đi trước)': 'Heuristic AB depth 3', 'Bên B (đi sau)': 'PMCS (100)', 'Số ván': 5, 'A thắng': 5, 'Hòa': 0, 'B thắng': 0, 'Thời gian TB/nước của Monte Carlo (ms)': 19.19}


In [36]:
results_table = pd.DataFrame(match_rows)
display(results_table)

# Biểu đồ: kết quả của Monte Carlo (thắng / hòa / thua) trong từng trận, tính theo phần trăm số ván.
labels, wins, draws, losses = [], [], [], []
for _, row in results_table.iterrows():
    mc_first = row["Monte Carlo đi trước"]
    w, l = (row["A thắng"], row["B thắng"]) if mc_first else (row["B thắng"], row["A thắng"])
    mc_name, opponent = (row["Bên A (đi trước)"], row["Bên B (đi sau)"]) if mc_first else (row["Bên B (đi sau)"], row["Bên A (đi trước)"])
    labels.append(f"{mc_name} {'đi trước' if mc_first else 'đi sau'}\nvs {opponent}, {row['Bàn']}")
    wins.append(100 * w / row["Số ván"]); draws.append(100 * row["Hòa"] / row["Số ván"]); losses.append(100 * l / row["Số ván"])

fig, ax = plt.subplots(figsize = (9, 0.6 * len(labels) + 1.5))
y = np.arange(len(labels))[::-1]
ax.barh(y, wins, color = "#2166ac", label = "Monte Carlo thắng", edgecolor = "white", linewidth = 2)
ax.barh(y, draws, left = wins, color = "#bdbdbd", label = "Hòa", edgecolor = "white", linewidth = 2)
ax.barh(y, losses, left = np.array(wins) + np.array(draws), color = "#d6604d", label = "Monte Carlo thua", edgecolor = "white", linewidth = 2)
ax.set_yticks(y); ax.set_yticklabels(labels, fontsize = 8)
ax.set_xlim(0, 100); ax.set_xlabel("% số ván")
ax.set_title("Kết quả của Pure Monte Carlo trong từng trận")
ax.legend(loc = "upper center", bbox_to_anchor = (0.5, -0.18), ncol = 3, frameon = False)
for spine in ("top", "right"): ax.spines[spine].set_visible(False)
plt.tight_layout()
plt.show()

,Trận,Bàn,Monte Carlo đi trước,Bên A (đi trước),Bên B (đi sau),Số ván,A thắng,Hòa,B thắng,Thời gian TB/nước của Monte Carlo (ms)
0,1. Monte Carlo (100) vs Alpha-Beta,4x4,True,PMCS (100),Alpha-Beta,20,0,9,11,10.97
1,1. Alpha-Beta vs Monte Carlo (100),4x4,False,Alpha-Beta,PMCS (100),20,3,17,0,9.66
2,1. Monte Carlo (1000) vs Alpha-Beta,4x4,True,PMCS (1000),Alpha-Beta,20,0,7,13,114.15
3,1. Alpha-Beta vs Monte Carlo (1000),4x4,False,Alpha-Beta,PMCS (1000),20,0,20,0,102.00
4,2. Monte Carlo (10) vs Random,6x7,True,PMCS (10),Random,100,81,0,19,2.19
5,2. Random vs Monte Carlo (10),6x7,False,Random,PMCS (10),100,29,0,71,1.99
6,2. Monte Carlo (100) vs Random,6x7,True,PMCS (100),Random,100,100,0,0,22.21
7,2. Random vs Monte Carlo (100),6x7,False,Random,PMCS (100),100,4,0,96,20.75
8,2. Monte Carlo (1000) vs Random,6x7,True,PMCS (1000),Random,20,20,0,0,222.69
9,2. Random vs Monte Carlo (1000),6x7,False,Random,PMCS (1000),20,0,0,20,226.15


**Kết quả các trận** (bảng và biểu đồ ở trên; seed `20261110`, `play` trả kết quả theo bên đi trước / đi sau).

* **Trận 1, bàn 4×4: Alpha-Beta không thua ván nào trong 80 ván** (có `assert`). Monte Carlo (N = 100) đi trước: 0 thắng, 9 hòa, 11 thua (20 ván); Alpha-Beta đi trước: Monte Carlo 0 thắng, 17 hòa, 3 thua. Monte Carlo (N = 1,000) đi trước: 0 thắng, 7 hòa, 13 thua; Alpha-Beta đi trước: 0 thắng, 20 hòa, 0 thua.
* **Trận 2, bàn 6×7 đấu Random:** N = 10 thắng 81/100 (đi trước) và 71/100 (đi sau); N = 100 thắng 100/100 và 96/100; N = 1,000 thắng 20/20 ở cả hai lượt. Thời gian TB mỗi nước của Monte Carlo khoảng 1.7 ms (N = 10), 20 - 26 ms (N = 100) và 210 - 260 ms (N = 1,000), tùy lần chạy.
* **Trận 3, đấu Heuristic Alpha-Beta của Tiến:** số thắng/hòa/thua khi thử PMCS (N=100) với cutoff 2 và 3 được tính trực tiếp ở bảng/biểu đồ phía trên (5 ván cho mỗi lượt đi). Kết quả của mẫu nhỏ này chỉ có tính minh họa; thời gian và kết quả phụ thuộc cấu hình, máy đo và seed.

### Which implementation plays stronger?

**Trên bàn nhỏ (4×4), Alpha-Beta mạnh hơn.** Alpha-Beta tìm tới hết ván nên chơi không sai, và nó không thua ván nào trong 80 ván. Monte Carlo không thắng nổi: nhiều nhất là hòa (7 - 20 ván hòa trên 20, tùy bên đi trước và N) và khi đi trước thì thua 11/20 (N = 100) và 13/20 (N = 1,000). Chú ý là tăng N từ 100 lên 1,000 **không làm Monte Carlo đỡ thua hơn** khi đi trước. Lý do: ván đánh thử cho đối thủ đi bừa, nên ước lượng điểm của Monte Carlo chỉ đúng khi đối thủ yếu. Đối thủ chơi đúng (Alpha-Beta) sẽ bắt bài những nước "có vẻ tốt" đó, nhiều ván thử hơn chỉ làm ước lượng chính xác hơn theo giả định sai đó, nên cũng không sửa được chuyện này.

**Trên bàn chuẩn 6×7, Alpha-Beta đầy đủ không chạy nổi** (Loạt 1 / Loạt 2 của Thiện ở Task 3: chỉ bàn trống 4×4 là xong; trên 6×7 đã vượt 500,000 nút khi còn 20 ô trống). Monte Carlo thì chạy được ngay: một nước đi với N = 1,000 khoảng 0.3 - 0.4 giây, và nó thắng Random 100/100 (N = 100, đi trước) và 20/20 (N = 1,000, hai lượt). Nhưng Random là đối thủ rất yếu, nên các con số này chưa nói được nhiều. Câu hỏi thật sự trên bàn 6×7 là Monte Carlo so với Heuristic Alpha-Beta (Task 4 của Tiến). Trận đối chứng cutoff 2 và 3 đã được bổ sung phía trên. Để đánh giá độ mạnh khách quan hơn, nên tăng số ván và đổi nhiều seed (vì Pure Monte Carlo có ngẫu nhiên).

**Mạnh hay yếu phụ thuộc vào:**

* **Thời gian cho phép mỗi nước:** Monte Carlo mạnh hơn khi tăng số ván `N` (N = 10 thắng 81/100, N = 100 thắng 100/100 khi đi trước); Heuristic Alpha-Beta thì tùy độ sâu cắt. Cả hai đều đổi thời gian lấy sức mạnh.
* **Chất lượng hàm chấm điểm** (với Heuristic Alpha-Beta): hàm chấm điểm tồi thì tìm sâu mấy cũng sai; Monte Carlo không cần hàm này, đó là ưu điểm của nó.
* **Đối thủ là ai:** Monte Carlo giả sử đối thủ đi ngẫu nhiên, nên dễ bị đối thủ giỏi bắt bài (trận 1), còn thắng Random rất dễ (trận 2).
* **Giai đoạn của ván:** đầu ván có rất nhiều khả năng nên 100 ván thử mỗi cột còn quá ít (điểm các cột giữa chỉ chênh vài phần trăm, xem ô minh họa); cuối ván ít nước hơn nên cả Alpha-Beta lẫn Monte Carlo đều chính xác hơn (bảng sáu bàn: N ≥ 100 đúng gần như 100%).

## Tổng kết Task 5

* **Chất lượng nước đi:** với N ≥ 100, Monte Carlo chọn đúng nước thắng ngay / nước phải chặn trong 239 / 240 lần thử (6 bàn × 20 lần × 2 mức N: N = 100 đúng 119/120, N = 1,000 đúng 120/120). N = 10 chỉ đúng khoảng một nửa (75/120).
* **Thời gian:** khoảng 0.3 - 0.4 ms mỗi ván đánh thử trên bàn 6×7; một nước đi N = 1,000 khoảng 0.3 - 0.4 giây (kể cả bàn trống 6×7), nên chạy được ngay trên bàn chuẩn.
* **So với Alpha-Beta:** Alpha-Beta luôn đúng và không thua ván nào trên 4×4, nhưng không chạy nổi trên 6×7 trống. Monte Carlo đấu Alpha-Beta trên 4×4 chỉ hòa hoặc thua.
* **Khả năng mở rộng:** thời gian tăng gần tỉ lệ thuận với `N` (không bùng nổ theo độ sâu), nên đổi sức mạnh lấy thời gian dễ dàng.

**Giới hạn.**

* Ván đánh thử là ngẫu nhiên, nên Monte Carlo ngầm giả sử đối thủ chơi bừa và dễ bị đối thủ giỏi bắt bài.
* Mỗi cột chỉ được xét bằng ván đánh thử: không nhìn sâu hơn một nước, và không đảm bảo tối ưu như Alpha-Beta.
* Kết quả có dao động theo seed: chạy lại với cùng seed ra đúng số này, đổi seed thì số đổi một chút (nhất là ở N nhỏ).
* Thời gian đo trên máy chạy notebook này, máy khác sẽ khác.
* So sánh với Heuristic Alpha-Beta của Tiến nay đã được thực hiện bằng thí nghiệm nhỏ trên 6×7 (cutoff 2 và 3). Không suy rộng kết quả 5 ván mỗi lượt đi thành đánh giá tổng quát.


## Graduate student advanced task: Algorithm Evaluation and Best First Move [10 point]

__Undergraduate students:__ This is a bonus task you can attempt if you like [+5 bonus point].

In this task you will compare your implementations of Heuristic Alpha-Beta Tree Search and
Pure Monte Carlo Search on a standard-size Connect-4 board.

### Parameter Tuning 

#### Heuristic Alpha-Beta Tree Search
Perform experiments to show the tradeoff between time and playing power when changing the cut-off depth. 
Things to think about:
* How can you measure performance?
* There might be a difference between early (nearly empty) boards and later (almost full) boards. 

Show the tradeoff in an appropriate graph.  

In [37]:
# Your code/ answer goes here.

#### Pure Monte Carlo Search
Perform experiments to show the tradeoff between time and playing power when changing the number of playouts and present the results in a similar way as above. 

In [38]:
# Your code/ answer goes here.

### Comparison

Compare the performance between the two implementations. Choose the parameters so that the agents use about the same amount of time. Present the results and discuss why you see these results.

In [39]:
# Your code/ answer goes here.